Temporal Semantic Novelty

Computes:
- Strict novelty  = 1 - max cosine similarity to prior work
- kNN novelty     = 1 - mean(top-k similarities to prior work)

Both computed in a single temporal pass.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

In [2]:
## Load embeddings and metadata
paper_ids = np.load('../../outputs/intermediate/paper_ids.npy')
embeddings = np.load('../../outputs/intermediate/abstract_embeddings.npy')

meta = pd.read_csv('../../outputs/intermediate/openalex_metadata_full.csv')

meta = meta[meta['global_paper_id'].isin(paper_ids)]
meta = meta.set_index('global_paper_id').loc[paper_ids].reset_index()

years = meta['year'].values

In [3]:
## Sort by year (temporal order)
sorted_idx = np.argsort(years)

embeddings = embeddings[sorted_idx]
paper_ids = paper_ids[sorted_idx]
years = years[sorted_idx]

In [4]:
## Compute Novelty
top_k = 5
n = len(embeddings)

semantic_strict = np.zeros(n)
semantic_knn = np.zeros(n)

for i in tqdm(range(n)):
    cur_emb = embeddings[i].reshape(1, -1)
    cur_years = years[i]

    prev_mask = years < cur_years

    if not prev_mask.any():
        semantic_strict[i] = 1.0
        semantic_knn[i] = 1.0
        continue

    prev_embeddings = embeddings[prev_mask]
    sims = cosine_similarity(cur_emb, prev_embeddings)[0]       # Calc. The sim.

    #Strict Novelty
    max_sim = np.max(sims)          # most similar prior paper
    semantic_strict[i] = 1-max_sim

    #kNN novelty
    k = min(top_k, len(sims))
    top_sim = np.partition(sims, -k)[-k:]
    semantic_knn[i] = np.mean(top_sim)

100%|██████████| 4242/4242 [00:26<00:00, 161.68it/s]


In [5]:
## Clip float values
semantic_strict = np.clip(semantic_strict, 0, 1)
semantic_knn = np.clip(semantic_knn, 0, 1)

In [6]:
print(semantic_strict)
print(semantic_knn)

[1.         1.         1.         ... 0.06365013 0.06365013 0.06365013]
[1.        1.        1.        ... 0.9248414 0.9248414 0.9248414]


In [7]:
## Save output
strict_df = pd.DataFrame({
    'paper_id': paper_ids,
    'year': years,
    'semantic_strict': semantic_strict
})

kNN_df = pd.DataFrame({
    'paper_id': paper_ids,
    'year': years,
    'semantic_knn': semantic_knn
})

strict_df.to_csv("../../outputs/final/semantic_novelty_scores.csv", index=False)
kNN_df.to_csv("../../outputs/final/semantic_novelty_knn_scores.csv", index=False)

In [10]:
scks = pd.read_csv('../../outputs/final/semantic_novelty_knn_scores.csv')
scks.describe()

,year,semantic_knn
count,4242.000000,4242.000000
mean,2019.798444,0.926361
std,3.327685,0.036648
min,2010.000000,0.570475
25%,2018.000000,0.917603
50%,2021.000000,0.933618
75%,2022.000000,0.943532
max,2025.000000,1.000000


In [12]:
scks['semantic_knn'].value_counts()

semantic_knn
1.000000    63
0.976205     8
0.927393     7
0.951873     7
0.921008     7
            ..
0.920725     1
0.907981     1
0.937626     1
0.921809     1
0.936839     1
Name: count, Length: 3509, dtype: int64